In [1]:
import pandas as pd
import unicodedata, re

pd.options.display.float_format = "{:,.2f}".format

In [2]:
#******************************************************************************************
# Primeira rotina: Tratamento e mapeamento das normalizações de nomes de campos e colunas
#******************************************************************************************
# ---------------- helpers: normalização de nomes ----------------
def _normalize_token(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))  # remove acento
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", "", s)  # só letras/números
    return s

def _build_mapper(groups):
    """
    groups = { 'Destino': ['sin1', 'sin2', ...], ... }  (os sinônimos podem estar acentuados)
    Retorna dict: token_normalizado -> 'Destino'
    """
    m = {}
    for dest, syns in groups.items():
        for s in syns:
            m[_normalize_token(s)] = dest
    return m

def canonicalize_columns(df: pd.DataFrame, groups: dict, required: list, where: str) -> pd.DataFrame:
    """
    - df: DataFrame original
    - groups: {'NomeDestino': [sinônimos...]}
    - required: lista de nomes destino obrigatórios
    - where: rótulo do DF (ex.: 'planos', 'custos', 'socios') para mensagens de erro
    """
    synmap = _build_mapper(groups)

    new_cols = {}
    for c in df.columns:
        tok = _normalize_token(c)
        if tok in synmap:
            new_cols[c] = synmap[tok]
        else:
            # fallback: usa o próprio nome em "Title Case"
            new_cols[c] = c.strip().title()

    df = df.rename(columns=new_cols)

    missing = [r for r in required if r not in df.columns]
    if missing:
        raise KeyError(
            f"[{where}] Colunas obrigatórias ausentes: {missing}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return df

#************************************************************************************
# Segunda rotina: Executando a Leitura dos arquivos (Planos e ofertas, Custos Fixos
# e Tratamento dos prolabores para os sócios)
#************************************************************************************

# ---------------- 1) leitura ----------------
df_planos = pd.read_csv("planos_acad.csv")
df_custos = pd.read_csv("custos_fixos_acad.csv")
df_socios = pd.read_csv("socios_acad.csv")

#********************************************************************************
# Terceira rotina: ------------ >>>> Normalização/mapeamento de colunas ---------
#********************************************************************************
PLANOS_GROUPS = {
    "Plano":   ["plano", "planos", "tipo", "categoria", "modalidade"],
    "Preco":   ["preco", "preço", "valor", "mensalidade", "contrato", "precoaluno", "valoraluno", "taxa", "tarifa"],
    "Alunos":  ["alunos", "qtdalunos", "qtdealunos", "qtd", "qtde", "quantidade", "aluno", "matriculados", "assinantes"],
}
df_planos = canonicalize_columns(df_planos, PLANOS_GROUPS, ["Plano","Preco","Alunos"], "planos")

#********************************************************************************
# Quarta rotina: Tratamento dos grupos dos custos envolvidos
#********************************************************************************
CUSTOS_GROUPS = {
    "Custo": ["custo", "despesa", "item", "descricao", "descrição"],
    "Valor": ["valor", "valor(r$)", "valor_rs", "valorrs", "valorreais", "valormensal", "montante"],
}
df_custos = canonicalize_columns(df_custos, CUSTOS_GROUPS, ["Custo","Valor"], "custos")

#************************************************************************************
# Quinta rotina: Readequação e normalização das palavras acentuadas e não acentuadas
#************************************************************************************
SOCIOS_GROUPS = {
    "Socio":      ["socio", "sócio", "socia", "sócia", "nome", "parceiro"],
    "Percentual": [
        "percentual", "perc", "porcentagem", "%", "percent", "percentagem",
        "participacao", "participação", "quota", "cota", "fracao", "fração",
        "part", "particip", "partic"
    ],
}

df_socios = canonicalize_columns(df_socios, SOCIOS_GROUPS, ["Socio","Percentual"], "socios")

#**************************************************************************************************
# Sexta rotina: Tratamento das colunas criando constantes para identificação de erros de conteúdo
#**************************************************************************************************
# ---------------- 1b) tipos - tratamento das colunas ----------------
df_planos["Preco"]  = pd.to_numeric(df_planos["Preco"], errors="coerce").fillna(0.0)
df_planos["Alunos"] = pd.to_numeric(df_planos["Alunos"], errors="coerce").fillna(0).astype(int)
df_custos["Valor"]  = pd.to_numeric(df_custos["Valor"], errors="coerce").fillna(0.0)
df_socios["Percentual"] = pd.to_numeric(df_socios["Percentual"], errors="coerce").fillna(0.0)

# Percentual para prolabore dos sócios - pode vir como 50 ou 0.5
df_socios["Frac"] = df_socios["Percentual"].apply(lambda x: x/100.0 if x > 1 else x)
total_frac = df_socios["Frac"].sum()
if total_frac > 0:
    df_socios["Frac"] = df_socios["Frac"] / total_frac
else:
    # sem sócios válidos: ninguém recebe pró-labore
    df_socios["Frac"] = 0.0
    
#*******************************************************************************************
# Sétima rotina: Montando o plano para vendas de pacotes de seções de exercícios na acadamia
#*******************************************************************************************
# ---------------- 2) receita mensal por plano ----------------
def mensalizar(plano: str, preco: float) -> float:
    p = (plano or "").strip().lower()
    if p == "semestral":
        return preco / 6.0
    elif p == "trimestral":
        return preco / 3.0
    elif p == "anual":
        return preco / 12.0
    else:
        return preco

receita_dados = []
for _, row in df_planos.iterrows():
    plano = str(row["Plano"]).strip().title()
    preco = float(row["Preco"])
    alunos = int(row["Alunos"])
    vm_aluno = mensalizar(plano, preco)
    rec_mensal = vm_aluno * alunos
    receita_dados.append([plano, alunos, preco, vm_aluno, rec_mensal])

df_receita = pd.DataFrame(
    receita_dados,
    columns=["Plano", "Qtde Alunos", "Contrato(R$)", "Valor Mensal por Aluno (R$)", "Receita Mensal (R$)"]
)
for col in ["Contrato(R$)", "Valor Mensal por Aluno (R$)", "Receita Mensal (R$)"]:
    df_receita[col] = df_receita[col].round(2)

receita_total = float(df_receita["Receita Mensal (R$)"].sum())

#************************************************************************************************
# Oitava rotina: definição da rotina de cálculos para apuração dos valores a serem apresentados
#************************************************************************************************
# ---------------- 3) custos e tributos ----------------
custo_total = float(df_custos["Valor"].sum())
tot_tributos = round(receita_total * 0.06, 2)

# ---------------- 4) pró-labore ----------------
lucro_operacional = max(receita_total - custo_total, 0.0)
base_prolab = round(lucro_operacional * 0.40, 2)
desc_inss = round(base_prolab * 0.11, 2)
desc_irpf = round(base_prolab * 0.14, 2)
val_prolab_liq = round((base_prolab - desc_inss - desc_irpf) + (receita_total / 5.0), 2)

if df_socios["Frac"].sum() > 0:
    dist_prolab = {row["Socio"]: round(val_prolab_liq * row["Frac"], 2) for _, row in df_socios.iterrows()}
else:
    dist_prolab = {}
tot_prolab = round(sum(dist_prolab.values()), 2)

# ---------------- 5) resultados ----------------
lucro_liquido = round((receita_total - custo_total) - (tot_prolab + tot_tributos), 2)
perc_liq = (lucro_liquido / receita_total * 100.0) if receita_total else 0.0
despesa_total = round(receita_total - lucro_liquido, 2)
perc_despesa_total = (despesa_total / receita_total * 100.0) if receita_total else 0.0



In [3]:
#***************************************************************************************************
# Nona rotina: Estabelecendo visões sobre os planos, valores e a temporalidade de suas aplicações
#***************************************************************************************************
# ---------------- 6) visões trimestral e anual ----------------
map_vm = {r["Plano"]: r["Valor Mensal por Aluno (R$)"] for _, r in df_receita.iterrows()}
map_qt = {r["Plano"]: r["Qtde Alunos"] for _, r in df_receita.iterrows()}
vm_mensal    = map_vm.get("Mensal", 0.0)
vm_trimestral= map_vm.get("Trimestral", 0.0)
vm_semestral = map_vm.get("Semestral", 0.0)
vm_anual     = map_vm.get("Anual", 0.0)
qa_mensal     = map_qt.get("Mensal", 0)
qa_trimestral = map_qt.get("Trimestral", 0)
qa_semestral  = map_qt.get("Semestral", 0)
qa_anual      = map_qt.get("Anual", 0)

fat_mes   = (vm_mensal     * qa_mensal)     * 3
fat_trim  = (vm_trimestral * qa_trimestral) * 3
fat_seme  = (vm_semestral  * qa_semestral)  * 3
fat_ano   = (vm_anual      * qa_anual)      * 3
total_trimestre = round(fat_mes + fat_trim + fat_seme + fat_ano, 2)

demo_mes   = (vm_mensal     * qa_mensal)     * 12
demo_trim  = (vm_trimestral * qa_trimestral) * 12
demo_seme  = (vm_semestral  * qa_semestral)  * 12
demo_ano   = (vm_anual      * qa_anual)      * 12
demo_rec_tot = round(demo_mes + demo_trim + demo_seme + demo_ano, 2)

In [4]:
#**************************************************************************************************
# Décima rotina: Apresentação dos dados financeiros apurados em função dos pacotes comercializados
#**************************************************************************************************
# ---------------- 7) impressão ----------------
print("="*86)
print(" ACADEMIAS - RELATÓRIO FINANCEIRO RESUMIDO")
print("="*86)

print("\n--- Receita por Plano ---")
print(df_receita[["Plano","Qtde Alunos","Contrato(R$)","Valor Mensal por Aluno (R$)","Receita Mensal (R$)"]]
      .sort_values("Plano").to_string(index=False))

print(f"Consolidado mensal ====================================================== >>> {receita_total:,.2f}")

print("")
print("=" * 86)
print("ACADEMIAS - DETALHAMENTO DO RELATÓRIO FINANCEIRO")
print("=" * 86)
print(f"Mensal:     Val_Aula = {vm_mensal:,.2f}     | Trimestre = {fat_mes:,.2f}   | Fatura/Mês = {vm_mensal*qa_mensal:,.2f}")
print(f"Trimestral: Val_Aula = {vm_trimestral:,.2f}     | Trimestre = {fat_trim:,.2f}   | Fatura/Mês = {vm_trimestral*qa_trimestral:,.2f}")
print(f"Semestral:  Val_Aula = {vm_semestral:,.2f}     | Trimestre = {fat_seme:,.2f}   | Fatura/Mês =  {vm_semestral*qa_semestral:,.2f}")
print(f"Anual:      Val_Aula = {vm_anual:,.2f}     | Trimestre = {fat_ano:,.2f}   | Fatura/Mês =  {vm_anual*qa_anual:,.2f}")
print("="*86)
print(f"Fat_Trimestral: --------------------------> R$ {total_trimestre:,.2f}   | Anual    R$ {demo_rec_tot:,.2f}")
print(" ")


 ACADEMIAS - RELATÓRIO FINANCEIRO RESUMIDO

--- Receita por Plano ---
     Plano  Qtde Alunos  Contrato(R$)  Valor Mensal por Aluno (R$)  Receita Mensal (R$)
     Anual           15      4,308.00                       359.00             5,385.00
    Mensal           40        423.00                       423.00            16,920.00
 Semestral           15      2,286.00                       381.00             5,715.00
Trimestral           30      1,200.00                       400.00            12,000.00
Consolidado mensal ====================================================== >>> 40,020.00

ACADEMIAS - DETALHAMENTO DO RELATÓRIO FINANCEIRO
Mensal:     Val_Aula = 423.00     | Trimestre = 50,760.00   | Fatura/Mês = 16,920.00
Trimestral: Val_Aula = 400.00     | Trimestre = 36,000.00   | Fatura/Mês = 12,000.00
Semestral:  Val_Aula = 381.00     | Trimestre = 17,145.00   | Fatura/Mês =  5,715.00
Anual:      Val_Aula = 359.00     | Trimestre = 16,155.00   | Fatura/Mês =  5,385.00
Fat_Trimestr

In [74]:
#*******************************************************************************************
# Décima primeira rotina: Apresentação dos resultados obtidos na operação do empreendimento
#*******************************************************************************************
print("=" * 86)
print("DEMONSTRATIVO DOS RESULTADOS")
print("=" * 86)
print(f"Receita Total Mensal:            R$ {receita_total:,.2f}")
print(f"Custos Fixos (Despesas Perman.): R$ {custo_total:,.2f}")
print(f"Tributos (6% s/ receita):        R$ {tot_tributos:,.2f}")
print("\n Pró-Labore                       Rubricas")
for nome, valor in dist_prolab.items():
    print(f"{nome:<30} R$ {valor:,.2f}")
print(f"Somatório dos Pró-labores:       R$ {tot_prolab:,.2f}")

print("\n--- Despesas & Resultado ---")
print(f"Despesas Totais Mensais:         R$ {despesa_total:,.2f}")
print(f"Perc. Despesa Total:             {perc_despesa_total:,.2f}%")
print("="*86)
print(f">>> Lucro Líquido Mensal:        R$ {lucro_liquido:,.2f}")
print(f">>> Lucro Líquido:               {perc_liq:,.2f}% da Receita Total Mensal")
print("="*86)

DEMONSTRATIVO DOS RESULTADOS
Receita Total Mensal:            R$ 40,020.00
Custos Fixos (Despesas Perman.): R$ 6,680.00
Tributos (6% s/ receita):        R$ 2,401.20

 Pró-Labore                       Rubricas
Socio1                         R$ 9,003.00
Socio2                         R$ 5,401.80
Socio3                         R$ 3,601.20
Somatório dos Pró-labores:       R$ 18,006.00

--- Despesas & Resultado ---
Despesas Totais Mensais:         R$ 27,087.20
Perc. Despesa Total:             67.68%
>>> Lucro Líquido Mensal:        R$ 12,932.80
>>> Lucro Líquido:               32.32% da Receita Total Mensal
